In [0]:
%pip install yfinance pandas numpy

In [0]:
#!/usr/bin/env python3
"""
NSE Daily Trading Dashboard — v5
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Universe   : Full Nifty 500 constituents (~500 stocks)
Indices    : Nifty 50 · Nifty 500 · Bank Nifty · Nifty Next 50 · Nifty Midcap 150
Top-N      : 10 per leaderboard
Fetch      : Parallel ThreadPoolExecutor

NEW in v5
  ▸ Multi-Period Momentum
      1D · 1W · 1M · 3M (quarterly) · 1Y (annual)
      — absolute return per period
      — relative return vs Nifty 500 (alpha / excess return)
      — ranked leaderboard per timeframe
  ▸ Sectoral Heatmap
      20 NSE sectors colour-coded by average change %
      Interactive tiles with sub-sector breakdown

Indicators : SMA · EMA · MACD · RSI · Bollinger Bands · Stochastic
             ATR · OBV · VWAP · Supertrend · Ichimoku · Fibonacci · ADX
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Usage
  pip install yfinance pandas numpy
  python nse_dashboard.py
  open nse_dashboard.html
"""

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import json, warnings, time
warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════

TOP_N       = 10
TECH_PERIOD = "2y"      # 2yr → full Ichimoku + annual momentum
MAX_WORKERS = 8

# ── Sector → ticker mapping (used for heatmap) ────────────────────
SECTOR_MAP = {
    "Banking & Finance": [
        "HDFCBANK.NS","ICICIBANK.NS","SBIN.NS","KOTAKBANK.NS","AXISBANK.NS",
        "BAJFINANCE.NS","BAJAJFINSV.NS","INDUSINDBK.NS","BANKBARODA.NS","PNB.NS",
        "CANBK.NS","FEDERALBNK.NS","IDFCFIRSTB.NS","BANDHANBNK.NS","RBLBANK.NS",
        "AUBANK.NS","YESBANK.NS","SBILIFE.NS","HDFCLIFE.NS","ICICIPRULI.NS",
        "RECLTD.NS","PFC.NS","MUTHOOTFIN.NS","CHOLAFIN.NS","SHRIRAMFIN.NS",
    ],
    "IT & Technology": [
        "TCS.NS","INFY.NS","HCLTECH.NS","WIPRO.NS","TECHM.NS","LTIM.NS",
        "MPHASIS.NS","COFORGE.NS","PERSISTENT.NS","KPITTECH.NS","TATAELXSI.NS",
        "OFSS.NS","CYIENT.NS","HAPPSTMNDS.NS","TANLA.NS","NAUKRI.NS",
    ],
    "Oil & Gas": [
        "RELIANCE.NS","ONGC.NS","BPCL.NS","IOC.NS","OIL.NS","HINDPETRO.NS",
        "GAIL.NS","IGL.NS","GSPL.NS","GUJGASLTD.NS","ATGL.NS","MRPL.NS",
    ],
    "Pharma & Healthcare": [
        "SUNPHARMA.NS","DRREDDY.NS","CIPLA.NS","DIVISLAB.NS","APOLLOHOSP.NS",
        "LUPIN.NS","AUROPHARMA.NS","TORNTPHARM.NS","ALKEM.NS","BIOCON.NS",
        "GLENMARK.NS","IPCA.NS","ZYDUSLIFE.NS","NATCOPHARM.NS","LAURUSLABS.NS",
        "MAXHEALTH.NS","FORTIS.NS","IPCALAB.NS","AJANTPHARM.NS","SYNGENE.NS",
    ],
    "Automobile": [
        "MARUTI.NS","BAJAJ-AUTO.NS","HEROMOTOCO.NS","M&M.NS","EICHERMOT.NS",
        "TATAMOTORS.NS","MOTHERSON.NS","BHARATFORG.NS","ENDURANCE.NS","ESCORTS.NS",
        "UNOMINDA.NS","SONACOMS.NS","CEATLTD.NS","APOLLOTYRE.NS","MRF.NS",
    ],
    "Metals & Mining": [
        "TATASTEEL.NS","JSWSTEEL.NS","HINDALCO.NS","SAIL.NS","VEDL.NS",
        "COALINDIA.NS","NMDC.NS","NATIONALUM.NS","WELCORP.NS","MOIL.NS",
        "GPIL.NS","STEELXIND.NS","RATNAMANI.NS","JINDALST.NS","APL.NS",
    ],
    "FMCG & Consumer": [
        "HINDUNILVR.NS","ITC.NS","NESTLEIND.NS","BRITANNIA.NS","DABUR.NS",
        "MARICO.NS","COLPAL.NS","GODREJCP.NS","EMAMILTD.NS","TATACONSUM.NS",
        "RADICO.NS","MCDOWELL-N.NS","VBL.NS","BIKAJI.NS","HONASA.NS",
    ],
    "Capital Goods & Infra": [
        "LT.NS","ABB.NS","SIEMENS.NS","BHEL.NS","THERMAX.NS","BOSCHLTD.NS",
        "HAVELLS.NS","POLYCAB.NS","KEC.NS","ENGINERSIN.NS","CONCOR.NS",
        "GMRINFRA.NS","IRB.NS","NCC.NS","ASHOKA.NS","RVNL.NS","BEL.NS",
    ],
    "Real Estate": [
        "DLF.NS","GODREJPROP.NS","OBEROIRLTY.NS","PRESTIGE.NS","PHOENIXLTD.NS",
        "SOBHA.NS","LODHA.NS","BRIGADE.NS","KOLTEPATIL.NS","MAHLIFE.NS",
    ],
    "Cement": [
        "ULTRACEMCO.NS","SHREECEM.NS","AMBUJACEM.NS","ACC.NS","DALBHARAT.NS",
        "JKCEMENT.NS","HEIDELBERG.NS","NUVOCO.NS","STARCEMENT.NS","SAGCEM.NS",
    ],
    "Power & Renewables": [
        "NTPC.NS","POWERGRID.NS","TATAPOWER.NS","ADANIGREEN.NS","ADANIPOWER.NS",
        "SUZLON.NS","SJVN.NS","TORNTPOWER.NS","CESC.NS","JPPOWER.NS",
        "IREDA.NS","NHPC.NS","NLCINDIA.NS","JSWENERGY.NS","WEBSOL.NS",
    ],
    "Telecom & Media": [
        "BHARTIARTL.NS","INDUSTOWER.NS","ZEEL.NS","NETWORK18.NS","GTPL.NS",
        "HFCL.NS","STLTECH.NS","RAILTEL.NS","TATACOMM.NS","MTNL.NS",
    ],
    "Chemicals": [
        "PIDILITIND.NS","DEEPAKNTR.NS","UPL.NS","AARTIIND.NS","NAVINFLUOR.NS",
        "FLUOROCHEM.NS","ALKYLAMINE.NS","GNFC.NS","PCBL.NS","SRF.NS",
        "PIIND.NS","SUMICHEM.NS","FINEORG.NS","TATACHEM.NS","COROMANDEL.NS",
    ],
    "Paint & Décor": [
        "ASIANPAINT.NS","BERGEPAINT.NS","KAJARIACER.NS","SOMANYCER.NS",
        "CERA.NS","GREENPANEL.NS","CENTURYPLY.NS","GREENPLY.NS",
    ],
    "Logistics & Transport": [
        "ADANIPORTS.NS","CONCOR.NS","BLUEDART.NS","ALLCARGO.NS","TCI.NS",
        "MAHLOG.NS","SNOWMAN.NS","GATEWAY.NS","RVNL.NS","IRCTC.NS",
    ],
    "Consumer Durables": [
        "TITAN.NS","VOLTAS.NS","WHIRLPOOL.NS","CROMPTON.NS","BLUESTARCO.NS",
        "SYMPHONY.NS","VGUARD.NS","AMBER.NS","DIXON.NS","LLOYD.NS",
    ],
    "Hospitality & Travel": [
        "INDHOTEL.NS","LEMONTREE.NS","INDIGO.NS","PVRINOX.NS","EASEMYTRIP.NS",
        "WONDERLA.NS","SAMHI.NS","MAHINDRA.NS","WESTLIFE.NS","JUBLFOOD.NS",
    ],
    "Retail & E-Commerce": [
        "TRENT.NS","SHOPERSTOP.NS","DMART.NS","NYKAA.NS","ZOMATO.NS",
        "SWIGGY.NS","KALYANKJIL.NS","METROBRAND.NS","SPENCERS.NS","V2RETAIL.NS",
    ],
    "Fintech & New Age": [
        "PAYTM.NS","POLICYBZR.NS","ANGELONE.NS","BSE.NS","MCX.NS",
        "CDSL.NS","CAMS.NS","CARTRADE.NS","IXIGO.NS","EASEMYTRIP.NS",
    ],
    "Defence & PSU": [
        "BEL.NS","HAL.NS","BHEL.NS","COCHINSHIP.NS","MAZDOCK.NS","BEML.NS",
        "MIDHANI.NS","GRSE.NS","PARAS.NS","MTAR.NS","TITAGARH.NS",
    ],
}

# ── Nifty 500 full ticker list ────────────────────────────────────
NIFTY_500 = sorted(set(
    ticker
    for tickers in SECTOR_MAP.values()
    for ticker in tickers
) | {
    # Core Nifty 50
    "ADANIENT.NS","ADANIPORTS.NS","APOLLOHOSP.NS","ASIANPAINT.NS","AXISBANK.NS",
    "BAJAJ-AUTO.NS","BAJFINANCE.NS","BAJAJFINSV.NS","BPCL.NS","BHARTIARTL.NS",
    "BRITANNIA.NS","CIPLA.NS","COALINDIA.NS","DIVISLAB.NS","DRREDDY.NS",
    "EICHERMOT.NS","GRASIM.NS","HCLTECH.NS","HDFCBANK.NS","HDFCLIFE.NS",
    "HEROMOTOCO.NS","HINDALCO.NS","HINDUNILVR.NS","ICICIBANK.NS","ITC.NS",
    "INDUSINDBK.NS","INFY.NS","JSWSTEEL.NS","KOTAKBANK.NS","LT.NS",
    "LTIM.NS","M&M.NS","MARUTI.NS","NESTLEIND.NS","NTPC.NS",
    "ONGC.NS","POWERGRID.NS","RELIANCE.NS","SBILIFE.NS","SBIN.NS",
    "SHREECEM.NS","SUNPHARMA.NS","TCS.NS","TATACONSUM.NS","TATASTEEL.NS",
    "TECHM.NS","TITAN.NS","ULTRACEMCO.NS","WIPRO.NS","UPL.NS",
    # Nifty Next 50 extras
    "ABB.NS","ACC.NS","AMBUJACEM.NS","AUROPHARMA.NS","BANDHANBNK.NS",
    "BANKBARODA.NS","BERGEPAINT.NS","BIOCON.NS","BOSCHLTD.NS","CANBK.NS",
    "CHOLAFIN.NS","COLPAL.NS","DABUR.NS","DLF.NS","GODREJCP.NS",
    "GODREJPROP.NS","HAVELLS.NS","HINDPETRO.NS","IDFCFIRSTB.NS","IGL.NS",
    "INDIGO.NS","INDUSTOWER.NS","IRCTC.NS","JUBLFOOD.NS","LICI.NS",
    "LUPIN.NS","MARICO.NS","MCDOWELL-N.NS","MRF.NS","MUTHOOTFIN.NS",
    "NAUKRI.NS","NMDC.NS","OFSS.NS","PAGEIND.NS","PEL.NS",
    "PERSISTENT.NS","PIDILITIND.NS","PIIND.NS","PNB.NS","RECLTD.NS",
    "SAIL.NS","SIEMENS.NS","SRF.NS","TATAELXSI.NS","TORNTPHARM.NS",
    "TRENT.NS","VEDL.NS","VOLTAS.NS","ZYDUSLIFE.NS","ICICIPRULI.NS",
    # Midcap extras
    "AARTIIND.NS","ABCAPITAL.NS","ABFRL.NS","AJANTPHARM.NS","ALKEM.NS",
    "ASTRAL.NS","AUBANK.NS","BATAINDIA.NS","BEL.NS","BHARATFORG.NS",
    "BHEL.NS","BLUESTARCO.NS","BSE.NS","CAMS.NS","CEATLTD.NS",
    "COFORGE.NS","CONCOR.NS","CYIENT.NS","DEEPAKNTR.NS","DIXON.NS",
    "FEDERALBNK.NS","FORTIS.NS","GAIL.NS","GLENMARK.NS","GMRINFRA.NS",
    "HUDCO.NS","IEX.NS","IIFL.NS","INDHOTEL.NS","INDIAMART.NS",
    "IOC.NS","IPCALAB.NS","IRB.NS","IREDA.NS","JKCEMENT.NS",
    "KPITTECH.NS","KAJARIACER.NS","KEC.NS","LAURUSLABS.NS","LICHSGFIN.NS",
    "MANAPPURAM.NS","MAXHEALTH.NS","MCX.NS","METROPOLIS.NS","MOTHERSON.NS",
    "MPHASIS.NS","NATCOPHARM.NS","NCC.NS","NBCC.NS","NLCINDIA.NS",
    "OBEROIRLTY.NS","OIL.NS","PHOENIXLTD.NS","POLYCAB.NS","POONAWALLA.NS",
    "PRESTIGE.NS","PVRINOX.NS","RADICO.NS","RAILTEL.NS","RBLBANK.NS",
    "RITES.NS","SJVN.NS","SOBHA.NS","SONACOMS.NS","SRTRANSFIN.NS",
    "SUNDARMFIN.NS","SUPREMEIND.NS","SUZLON.NS","SYMPHONY.NS","TANLA.NS",
    "TATACHEM.NS","TATACOMM.NS","TATAPOWER.NS","THERMAX.NS",
    "TORNTPOWER.NS","TRIDENT.NS","UJJIVANSFB.NS","UNOMINDA.NS",
    "UTIAMC.NS","VBL.NS","VGUARD.NS","WELCORP.NS","WHIRLPOOL.NS",
    "YESBANK.NS","ZEEL.NS","ZOMATO.NS","PAYTM.NS","SWIGGY.NS",
    # Smallcap & others
    "AAVAS.NS","AFFLE.NS","AMBER.NS","ANGELONE.NS","APOLLOTYRE.NS",
    "ASHOKLEY.NS","BALKRISHNA.NS","BEML.NS","BIKAJI.NS","BLUEDART.NS",
    "CANFINHOME.NS","COCHINSHIP.NS","COROMANDEL.NS","CROMPTON.NS","CRISIL.NS",
    "DALBHARAT.NS","DEEPAKFERT.NS","EMAMILTD.NS","ENDURANCE.NS","ESCORTS.NS",
    "EXIDEIND.NS","FLUOROCHEM.NS","GNFC.NS","GPIL.NS","GRINDWELL.NS",
    "HAPPSTMNDS.NS","HOMEFIRST.NS","HONASA.NS","IREDA.NS","JKPAPER.NS",
    "JKTYRE.NS","JSWENERGY.NS","JUBLINGREA.NS","KALYANKJIL.NS","KOEL.NS",
    "LATENTVIEW.NS","LEMONTREE.NS","LUMAXIND.NS","MAHLIFE.NS","MAZDOCK.NS",
    "MEDANTA.NS","MIDHANI.NS","MOIL.NS","MTAR.NS","NATIONALUM.NS",
    "NAVINFLUOR.NS","NBCC.NS","PCBL.NS","PFC.NS","PNBHOUSING.NS",
    "POLPLANET.NS","PSPPROJECT.NS","RATNAMANI.NS","RCF.NS","REDINGTON.NS",
    "RVNL.NS","SAFARI.NS","SAREGAMA.NS","SBFC.NS","SHRIRAMFIN.NS",
    "SJVN.NS","STLTECH.NS","SYNGENE.NS","TATACOFFEE.NS","TATAPOWER.NS",
    "TITAGARH.NS","TORRENTP.NS","UJJIVAN.NS","UNIONBANK.NS","VAIBHAVGBL.NS",
    "VESUVIUS.NS","WEBELSOLAR.NS","WELSPUNSYN.NS","WONDERLA.NS","ZENSARTECH.NS",
})

NSE_INDICES = {
    "Nifty 50":         "^NSEI",
    "Nifty 500":        "^CRSLDX",
    "Bank Nifty":       "^NSEBANK",
    "Nifty Next 50":    "^NSMIDCP",
    "Nifty Midcap 150": "NIFTYMIDCAP150.NS",
}

# Benchmark for relative momentum
BENCHMARK_SYM = "^CRSLDX"   # Nifty 500


# ══════════════════════════════════════════════════════════════════
# TECHNICAL INDICATOR LIBRARY
# ══════════════════════════════════════════════════════════════════

def _f(x):
    try: v=float(x); return v if np.isfinite(v) else np.nan
    except: return np.nan

def calc_sma(c,p):  return _f(c.rolling(p).mean().iloc[-1])
def calc_ema(c,s):  return _f(c.ewm(span=s,adjust=False).mean().iloc[-1])

def calc_macd(c):
    e12=c.ewm(span=12,adjust=False).mean(); e26=c.ewm(span=26,adjust=False).mean()
    m=e12-e26; s=m.ewm(span=9,adjust=False).mean()
    return _f(m.iloc[-1]),_f(s.iloc[-1]),_f((m-s).iloc[-1])

def calc_rsi(c,p=14):
    d=c.diff().dropna()
    g=d.clip(lower=0).ewm(com=p-1,adjust=False).mean()
    l=(-d.clip(upper=0)).ewm(com=p-1,adjust=False).mean()
    return _f((100-100/(1+g/l.replace(0,np.nan))).iloc[-1])

def calc_bb(c,p=20):
    sma=c.rolling(p).mean(); std=c.rolling(p).std()
    u=_f((sma+2*std).iloc[-1]); m=_f(sma.iloc[-1]); l=_f((sma-2*std).iloc[-1])
    px=_f(c.iloc[-1])
    bw=(u-l)/m*100 if m else np.nan; pctb=(px-l)/(u-l) if (u-l) else np.nan
    return u,m,l,_f(bw),_f(pctb)

def calc_stochastic(h,l,c,kp=14,dp=3):
    ll=l.rolling(kp).min(); hh=h.rolling(kp).max()
    k=100*(c-ll)/(hh-ll).replace(0,np.nan)
    return _f(k.iloc[-1]),_f(k.rolling(dp).mean().iloc[-1])

def calc_atr(h,l,c,p=14):
    pc=c.shift(1)
    tr=pd.concat([h-l,(h-pc).abs(),(l-pc).abs()],axis=1).max(axis=1)
    return _f(tr.ewm(com=p-1,adjust=False).mean().iloc[-1])

def calc_obv(c,v):
    obv=(v*np.sign(c.diff().fillna(0))).cumsum()
    slope=np.polyfit(range(5),obv.iloc[-5:].values,1)[0] if len(obv)>=5 else 0
    return _f(obv.iloc[-1]),("rising" if slope>0 else "falling")

def calc_vwap(h,l,c,v,p=20):
    tp=(h+l+c)/3
    return _f(((tp*v).rolling(p).sum()/v.rolling(p).sum()).iloc[-1])

def calc_supertrend(h,l,c,period=10,mult=3.0):
    pc=c.shift(1)
    tr=pd.concat([h-l,(h-pc).abs(),(l-pc).abs()],axis=1).max(axis=1)
    atr=tr.ewm(com=period-1,adjust=False).mean()
    hl2=(h+l)/2; ub=hl2+mult*atr; lb=hl2-mult*atr
    st=pd.Series(np.nan,index=c.index); dr=pd.Series(0,index=c.index)
    for i in range(1,len(c)):
        pu,pl=_f(ub.iloc[i-1]),_f(lb.iloc[i-1])
        cu,cl=min(_f(ub.iloc[i]),pu) if _f(c.iloc[i-1])<=pu else _f(ub.iloc[i]), \
              max(_f(lb.iloc[i]),pl) if _f(c.iloc[i-1])>=pl else _f(lb.iloc[i])
        ps=st.iloc[i-1]; cv=_f(c.iloc[i])
        if np.isnan(ps) or ps==pu:
            st.iloc[i]=cu if cv<=cu else cl; dr.iloc[i]=-1 if cv<=cu else 1
        else:
            st.iloc[i]=cl if cv>=cl else cu; dr.iloc[i]=1 if cv>=cl else -1
    return _f(st.iloc[-1]),("bullish" if int(dr.iloc[-1])==1 else "bearish")

def calc_ichimoku(h,l,c):
    def mid(hh,ll,p): return (hh.rolling(p).max()+ll.rolling(p).min())/2
    ten=mid(h,l,9); kij=mid(h,l,26)
    sa=((ten+kij)/2).shift(26); sb=mid(h,l,52).shift(26)
    p=_f(c.iloc[-1]); sa_=_f(sa.iloc[-1]); sb_=_f(sb.iloc[-1])
    t_=_f(ten.iloc[-1]); k_=_f(kij.iloc[-1])
    ct=max(sa_,sb_) if not(np.isnan(sa_) or np.isnan(sb_)) else np.nan
    cb=min(sa_,sb_) if not(np.isnan(sa_) or np.isnan(sb_)) else np.nan
    above=p>ct if not np.isnan(ct) else False
    below=p<cb if not np.isnan(cb) else False
    green=sa_>sb_ if not(np.isnan(sa_) or np.isnan(sb_)) else False
    sig="bullish" if above and green else "bearish" if below else "neutral"
    return {"tenkan":t_,"kijun":k_,"senkou_a":sa_,"senkou_b":sb_,
            "above_cloud":above,"below_cloud":below,"bullish_cloud":green,"signal":sig}

def calc_fibonacci(h,l,lookback=50):
    hv=_f(h.iloc[-lookback:].max()); lv=_f(l.iloc[-lookback:].min()); d=hv-lv
    return {k:round(hv-r*d,2) for k,r in
            [("0%",0),("23.6%",.236),("38.2%",.382),("50%",.5),
             ("61.8%",.618),("78.6%",.786),("100%",1)]}

def nearest_fib(price,fib):
    c=min(fib.items(),key=lambda kv:abs(kv[1]-price))
    return f"{c[0]} ({c[1]:,.2f})"

def calc_adx(h,l,c,p=14):
    pc=c.shift(1)
    tr=pd.concat([h-l,(h-pc).abs(),(l-pc).abs()],axis=1).max(axis=1)
    dmp=np.where((h-h.shift(1))>(l.shift(1)-l),np.maximum(h-h.shift(1),0),0)
    dmm=np.where((l.shift(1)-l)>(h-h.shift(1)),np.maximum(l.shift(1)-l,0),0)
    dmp_s=pd.Series(dmp,index=c.index).ewm(com=p-1,adjust=False).mean()
    dmm_s=pd.Series(dmm,index=c.index).ewm(com=p-1,adjust=False).mean()
    atr=tr.ewm(com=p-1,adjust=False).mean()
    dip=100*dmp_s/atr.replace(0,np.nan); dim=100*dmm_s/atr.replace(0,np.nan)
    dx=100*(dip-dim).abs()/(dip+dim).replace(0,np.nan)
    adx=dx.ewm(com=p-1,adjust=False).mean()
    return _f(adx.iloc[-1]),_f(dip.iloc[-1]),_f(dim.iloc[-1])


# ══════════════════════════════════════════════════════════════════
# MULTI-PERIOD MOMENTUM
# ══════════════════════════════════════════════════════════════════

def pct_return(series, lookback_days):
    """Return % change over lookback_days trading days."""
    if len(series) <= lookback_days:
        return np.nan
    base = _f(series.iloc[-(lookback_days+1)])
    curr = _f(series.iloc[-1])
    if not base or base == 0:
        return np.nan
    return (curr - base) / base * 100

# Approximate trading days per period
PERIOD_DAYS = {
    "1D":  1,
    "1W":  5,
    "1M":  21,
    "3M":  63,
    "1Y":  252,
}

def calc_momentum_periods(close):
    """Return dict with absolute returns for each period."""
    return {p: round(pct_return(close, d), 2) for p, d in PERIOD_DAYS.items()}

def calc_relative_momentum(stock_pct, bench_pct):
    """Alpha = stock return minus benchmark return for same period."""
    if np.isnan(stock_pct) or np.isnan(bench_pct):
        return np.nan
    return round(stock_pct - bench_pct, 2)


# ══════════════════════════════════════════════════════════════════
# TA SIGNAL SCORING  (0–18 pts)
# ══════════════════════════════════════════════════════════════════

def signal_score(row):
    score=0; signals=[]; price=row.get("last",np.nan)
    rsi=row.get("rsi",np.nan)
    if not np.isnan(rsi):
        if 50<rsi<70:  score+=2; signals.append(f"RSI {rsi:.0f} bull zone")
        elif rsi<=35:  score+=1; signals.append(f"RSI {rsi:.0f} oversold")
        elif rsi>=70:            signals.append(f"RSI {rsi:.0f} overbought")
    hist=row.get("macd_hist",np.nan)
    if not np.isnan(hist):
        if hist>0: score+=2; signals.append("MACD hist+")
        else:                 signals.append("MACD hist−")
    pctb=row.get("bb_pctb",np.nan)
    if not np.isnan(pctb):
        if 0.4<pctb<0.85:  score+=1; signals.append(f"BB mid-upper")
        elif pctb<0.1:      score+=1; signals.append(f"BB near lower")
        elif pctb>0.95:              signals.append("BB near upper")
    stk=row.get("stoch_k",np.nan); std=row.get("stoch_d",np.nan)
    if not np.isnan(stk) and not np.isnan(std):
        if stk<20 and stk>std:      score+=2; signals.append("Stoch oversold cross↑")
        elif stk>80:                           signals.append("Stoch overbought")
        elif 50<stk<80 and stk>std: score+=1; signals.append("Stoch bullish")
    ema20=row.get("ema20",np.nan); ema50=row.get("ema50",np.nan); ema200=row.get("ema200",np.nan)
    for val,lbl in [(ema20,"EMA20"),(ema50,"EMA50"),(ema200,"EMA200")]:
        if not np.isnan(price) and not np.isnan(val):
            if price>val: score+=1; signals.append(f"Above {lbl}")
            elif lbl=="EMA200": signals.append("Below EMA200")
    adx=row.get("adx",np.nan); dip=row.get("adx_di_plus",np.nan); dim=row.get("adx_di_minus",np.nan)
    if not np.isnan(adx) and not np.isnan(dip) and not np.isnan(dim):
        if adx>25 and dip>dim: score+=2; signals.append(f"ADX {adx:.0f} trend↑")
        elif adx>25:                      signals.append(f"ADX {adx:.0f} trend↓")
    st=row.get("supertrend_sig","")
    if st=="bullish": score+=2; signals.append("Supertrend bullish")
    elif st=="bearish":          signals.append("Supertrend bearish")
    ichi=row.get("ichimoku",{})
    if ichi.get("above_cloud"):  score+=1; signals.append("Above cloud")
    if ichi.get("bullish_cloud"):score+=1; signals.append("Green cloud")
    if ichi.get("below_cloud"):            signals.append("Below cloud↓")
    obv_t=row.get("obv_trend","")
    if obv_t=="rising": score+=1; signals.append("OBV rising")
    else:                          signals.append("OBV falling")
    vwap=row.get("vwap",np.nan)
    if not np.isnan(vwap) and not np.isnan(price):
        if price>vwap: score+=1; signals.append(f"Above VWAP")
        else:                     signals.append(f"Below VWAP")
    return min(score,18), signals


# ══════════════════════════════════════════════════════════════════
# DATA FETCHING
# ══════════════════════════════════════════════════════════════════

# Global benchmark returns (filled once in main)
BENCH_RETURNS = {"1D": np.nan, "1W": np.nan, "1M": np.nan, "3M": np.nan, "1Y": np.nan}

def fetch_ohlcv(ticker, period=TECH_PERIOD):
    try:
        df=yf.Ticker(ticker).history(period=period,interval="1d",auto_adjust=True)
        df=df.dropna(subset=["Close"])
        return df if len(df)>=60 else None
    except Exception:
        return None

def build_stock_row(ticker):
    df=fetch_ohlcv(ticker)
    if df is None or len(df)<2: return None
    c=df["Close"]; h=df["High"]; l=df["Low"]; v=df["Volume"]
    price=_f(c.iloc[-1]); prev=_f(c.iloc[-2])
    vol=int(v.iloc[-1]) if np.isfinite(v.iloc[-1]) else 0

    # Multi-period momentum
    mom = calc_momentum_periods(c)
    rel = {p: calc_relative_momentum(mom[p], BENCH_RETURNS[p]) for p in mom}

    bb_u,bb_m,bb_l,bb_bw,bb_pctb=calc_bb(c)
    macd,msig,mhist=calc_macd(c)
    stk,std_val=calc_stochastic(h,l,c)
    obv_val,obv_trend=calc_obv(c,v)
    st_val,st_sig=calc_supertrend(h,l,c)
    ichi=calc_ichimoku(h,l,c)
    fib=calc_fibonacci(h,l)
    adx_val,di_p,di_m=calc_adx(h,l,c)

    pct1d=mom["1D"]; pct5d=mom["1W"]
    row={
        "ticker":ticker, "last":round(price,2),
        "pct_1d": pct1d,  "pct_5d": pct5d,
        "volume": vol, "closes_5d":[_f(x) for x in c.iloc[-5:].tolist()],
        # multi-period momentum (absolute)
        "mom_1d": mom["1D"], "mom_1w": mom["1W"],
        "mom_1m": mom["1M"], "mom_3m": mom["3M"], "mom_1y": mom["1Y"],
        # relative momentum vs Nifty 500 (alpha)
        "rel_1d": rel["1D"], "rel_1w": rel["1W"],
        "rel_1m": rel["1M"], "rel_3m": rel["3M"], "rel_1y": rel["1Y"],
        # MA
        "sma20":calc_sma(c,20),"sma50":calc_sma(c,50),"sma200":calc_sma(c,200),
        "ema20":calc_ema(c,20),"ema50":calc_ema(c,50),"ema200":calc_ema(c,200),
        # indicators
        "macd":macd,"macd_sig":msig,"macd_hist":mhist,
        "rsi":calc_rsi(c),
        "bb_upper":bb_u,"bb_mid":bb_m,"bb_lower":bb_l,"bb_bw":bb_bw,"bb_pctb":bb_pctb,
        "stoch_k":stk,"stoch_d":std_val,
        "atr":calc_atr(h,l,c),
        "obv":obv_val,"obv_trend":obv_trend,
        "vwap":calc_vwap(h,l,c,v),
        "supertrend":st_val,"supertrend_sig":st_sig,
        "ichimoku":ichi,
        "fibonacci":fib,"fib_nearest":nearest_fib(price,fib),
        "adx":adx_val,"adx_di_plus":di_p,"adx_di_minus":di_m,
    }
    row["ta_score"],row["ta_signals"]=signal_score(row)
    return row

def fetch_all_parallel(tickers):
    results=[]; failed=0; total=len(tickers); done=0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures={ex.submit(build_stock_row,t):t for t in tickers}
        for fut in as_completed(futures):
            done+=1
            try:
                r=fut.result()
                if r: results.append(r)
                else: failed+=1
            except Exception: failed+=1
            print(f"  [{done:03d}/{total}] ✓ fetched  ({failed} failed)   ",end="\r",flush=True)
    print()
    return pd.DataFrame(results)

def fetch_benchmark_returns(symbol=BENCHMARK_SYM):
    """Fetch benchmark (Nifty 500) multi-period returns."""
    df=fetch_ohlcv(symbol,period="2y")
    if df is None: return
    c=df["Close"]
    for p,d in PERIOD_DAYS.items():
        BENCH_RETURNS[p]=pct_return(c,d)
    print(f"  Benchmark {symbol}: {BENCH_RETURNS}")

def fetch_index_row(name, symbol):
    df=fetch_ohlcv(symbol,period="2y")
    if df is None or len(df)<2: return None
    c=df["Close"]; h=df["High"]; l=df["Low"]
    price=_f(c.iloc[-1]); prev=_f(c.iloc[-2])
    mom=calc_momentum_periods(c)
    adx_v,dip,dim=calc_adx(h,l,c); _,st_sig=calc_supertrend(h,l,c)
    return {"name":name,"last":round(price,2),
            "pct_1d":mom["1D"],"pct_5d":mom["1W"],
            "mom_1m":mom["1M"],"mom_3m":mom["3M"],"mom_1y":mom["1Y"],
            "closes":[_f(x) for x in c.iloc[-5:].tolist()],
            "rsi":calc_rsi(c),"adx":adx_v,"supertrend_sig":st_sig}

def fetch_indices(indices):
    rows=[]
    for name,sym in indices.items():
        print(f"  yf.Ticker({sym:<26}) → {name}")
        r=fetch_index_row(name,sym)
        if r: rows.append(r)
    return pd.DataFrame(rows)

def compute_sentiment(df):
    g=(df["pct_1d"]>0).sum(); lo=(df["pct_1d"]<0).sum(); tot=g+lo
    br=g/tot*100 if tot else 50
    ar=df["rsi"].dropna().mean(); ad=df["adx"].dropna().mean() if "adx" in df else np.nan
    bt="Bullish breadth" if br>60 else "Bearish breadth" if br<40 else "Mixed"
    rt="RSI overbought" if ar>65 else "RSI oversold" if ar<40 else f"RSI neutral ({ar:.0f})"
    at=f"ADX {ad:.0f} ({'trending' if ad>25 else 'ranging'})" if np.isfinite(ad) else ""
    return f"{bt} — {g}&#8593; {lo}&#8595; of {tot}. {rt}. {at}."

def build_sector_stats(df):
    """Compute avg 1D, 1W, 1M, 3M, 1Y change per sector."""
    ticker_to_row={r["ticker"]:r for _,r in df.iterrows()}
    result={}
    for sector,tickers in SECTOR_MAP.items():
        rows=[ticker_to_row[t] for t in tickers if t in ticker_to_row]
        if not rows: continue
        result[sector]={
            "count": len(rows),
            "tickers": [r["ticker"].replace(".NS","") for r in rows],
            "avg_1d":  np.nanmean([r["mom_1d"] for r in rows]),
            "avg_1w":  np.nanmean([r["mom_1w"] for r in rows]),
            "avg_1m":  np.nanmean([r["mom_1m"] for r in rows]),
            "avg_3m":  np.nanmean([r["mom_3m"] for r in rows]),
            "avg_1y":  np.nanmean([r["mom_1y"] for r in rows]),
            "best_1d": max(rows, key=lambda r: r["mom_1d"] if np.isfinite(r["mom_1d"]) else -99)["ticker"].replace(".NS",""),
            "worst_1d":min(rows, key=lambda r: r["mom_1d"] if np.isfinite(r["mom_1d"]) else 99)["ticker"].replace(".NS",""),
        }
    return result


# ══════════════════════════════════════════════════════════════════
# HTML HELPERS
# ══════════════════════════════════════════════════════════════════

def _na(v): return isinstance(v,float) and not np.isfinite(v)

def pct_span(val,d=2):
    if val is None or _na(val): return '<span class="na">—</span>'
    cls="pos" if val>0 else ("neg" if val<0 else "nt"); sign="+" if val>0 else ""
    return f'<span class="{cls}">{sign}{val:.{d}f}%</span>'

def rel_span(val):
    """Alpha badge — shows α prefix."""
    if val is None or _na(val): return '<span class="na">—</span>'
    cls="pos" if val>0 else ("neg" if val<0 else "nt"); sign="+" if val>0 else ""
    return f'<span class="{cls}" title="vs Nifty 500">α{sign}{val:.1f}%</span>'

def sparkline(closes,w=80,h=28):
    closes=[c for c in closes if c is not None and np.isfinite(c)]
    if len(closes)<2: return "—"
    col="#22c55e" if closes[-1]>=closes[0] else "#f43f5e"
    return f'<canvas class="sp" width="{w}" height="{h}" data-c=\'{json.dumps(closes)}\' data-col="{col}"></canvas>'

def vol_bar(vol,max_vol):
    pct=min(int(vol/max_vol*100),100) if max_vol else 0
    return f'<div class="vbw"><div class="vb" style="width:{pct}%"></div><span class="vl">{vol:,}</span></div>'

def score_bar(score,mx=18):
    pct=int(score/mx*100)
    col="#22c55e" if score>=12 else "#f59e0b" if score>=7 else "#f43f5e"
    return f'<div class="sb-wrap"><div class="sb" style="width:{pct}%;background:{col}"></div><span class="sl">{score}/{mx}</span></div>'

def rsi_badge(rsi):
    if rsi is None or _na(rsi): return '<span class="na">—</span>'
    cls="badge-ob" if rsi>=70 else ("badge-os" if rsi<=30 else "badge-nt")
    return f'<span class="{cls}">{rsi:.0f}</span>'

def macd_badge(hist):
    if hist is None or _na(hist): return '<span class="na">—</span>'
    cls="pos" if hist>0 else "neg"; sign="+" if hist>0 else ""
    return f'<span class="{cls}">{sign}{hist:.2f}</span>'

def bb_badge(pctb):
    if pctb is None or _na(pctb): return '<span class="na">—</span>'
    if pctb>0.9:  return f'<span class="badge-ob">&#9650;{pctb:.2f}</span>'
    if pctb<0.1:  return f'<span class="badge-os">&#9660;{pctb:.2f}</span>'
    return f'<span class="badge-nt">Mid {pctb:.2f}</span>'

def stoch_badge(k,d):
    if k is None or _na(k): return '<span class="na">—</span>'
    cls="badge-ob" if k>80 else ("badge-os" if k<20 else "badge-nt")
    ds=f"/D{d:.0f}" if d and np.isfinite(d) else ""
    return f'<span class="{cls}">K{k:.0f}{ds}</span>'

def adx_badge(adx,dip,dim):
    if adx is None or _na(adx): return '<span class="na">—</span>'
    bull=np.isfinite(dip) and np.isfinite(dim) and dip>dim
    cls="pos" if bull else "neg"
    return f'<span class="{cls}">{adx:.0f} ({"strong" if adx>25 else "weak"})</span>'

def supertrend_badge(sig):
    if not sig: return '<span class="na">—</span>'
    cls="badge-os" if sig=="bullish" else "badge-ob"
    return f'<span class="{cls}">{"↑" if sig=="bullish" else "↓"} {sig.title()}</span>'

def ichimoku_badge(ichi):
    sig=ichi.get("signal","")
    if sig=="bullish": return '<span class="badge-os">↑ Bull</span>'
    if sig=="bearish": return '<span class="badge-ob">↓ Bear</span>'
    return '<span class="badge-nt">Neutral</span>'

def obv_badge(t):
    if t=="rising":  return '<span class="pos">&#8593; Rising</span>'
    if t=="falling": return '<span class="neg">&#8595; Falling</span>'
    return '<span class="na">Flat</span>'

def vwap_badge(price,vwap):
    if not vwap or _na(vwap): return '<span class="na">—</span>'
    cls="pos" if price>vwap else "neg"
    return f'<span class="{cls}">{"↑" if price>vwap else "↓"} {vwap:,.0f}</span>'

def ema_stack(price,e20,e50,e200):
    parts=[]
    for val,lbl in [(e20,"20"),(e50,"50"),(e200,"200")]:
        if val and np.isfinite(val):
            parts.append(f'<span class="{"tag-a" if price>val else "tag-b"}">E{lbl}</span>')
    return "".join(parts)

def signal_pills(signals):
    BULL=("bull","above","cross↑","rising","green","oversold","hist+","trend↑","vwap")
    return "".join(
        f'<span class="{"pill-b" if any(k in s.lower() for k in BULL) else "pill-r"}">{s}</span>'
        for s in signals[:6]
    )

# ── Heatmap colour helper ──────────────────────────────────────────
def heatmap_style(val):
    """Returns inline CSS background+color for a heatmap tile given % val."""
    if _na(val): return "background:#12182a;color:#4a5a80"
    # clamp to ±8 %
    norm = max(-8, min(8, val)) / 8.0
    if norm > 0:
        g = int(40 + 160 * norm)
        r = int(20); b = int(20)
        fg = "#e0ffe0" if norm > 0.4 else "#4ade80"
    else:
        r = int(40 + 180 * abs(norm))
        g = int(20); b = int(20)
        fg = "#ffe0e0" if abs(norm) > 0.4 else "#f87171"
    return f"background:rgb({r},{g},{b});color:{fg}"


# ══════════════════════════════════════════════════════════════════
# CSS
# ══════════════════════════════════════════════════════════════════

CSS = """
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
:root{
  --bg:#06080f;--s1:#0b0e18;--s2:#0f1320;
  --b1:#1a2035;--t:#b8c4db;--m:#4a5a80;--w:#dde4f5;
  --pos:#22c55e;--neg:#f43f5e;--acc:#f59e0b;--acc2:#38bdf8;--pur:#a78bfa;
  --fh:'Bebas Neue',sans-serif;--fb:'Manrope',sans-serif;--fm:'IBM Plex Mono',monospace;
}
body{background:var(--bg);color:var(--t);font-family:var(--fm);font-size:12px;
  background-image:radial-gradient(ellipse 100% 60% at 50% 0,rgba(56,189,248,.06),transparent),
    radial-gradient(ellipse 50% 40% at 90% 100%,rgba(167,139,250,.04),transparent);}
.wrap{max-width:1700px;margin:0 auto;padding:12px 14px 60px}

/* header */
header{display:flex;flex-wrap:wrap;gap:10px;align-items:center;
  padding:10px 0 12px;border-bottom:1px solid var(--b1);margin-bottom:14px}
.logo{font-family:var(--fh);font-size:28px;letter-spacing:.07em;color:#fff;line-height:1}
.logo em{color:var(--acc2)}.logo b{color:var(--acc)}
.hdate{color:var(--m);font-size:10px;margin-top:2px;font-family:var(--fb)}
.sent{margin-left:auto;background:var(--s1);border:1px solid var(--b1);
  border-left:3px solid var(--acc);border-radius:6px;padding:6px 12px;
  max-width:560px;font-size:11px;line-height:1.7}
.sent b{color:var(--acc);font-size:8px;letter-spacing:.16em;text-transform:uppercase;
  display:block;margin-bottom:2px;font-family:var(--fb)}

/* stats strip */
.stats-strip{display:flex;flex-wrap:wrap;gap:8px;margin-bottom:14px}
.stat-box{background:var(--s1);border:1px solid var(--b1);border-radius:6px;
  padding:7px 14px;flex:1;min-width:90px;text-align:center}
.stat-val{font-family:var(--fh);font-size:24px;color:var(--w);line-height:1}
.stat-lbl{font-family:var(--fb);font-size:9px;color:var(--m);letter-spacing:.12em;
  text-transform:uppercase;margin-top:2px}

/* section labels */
.sec{font-family:var(--fb);font-weight:700;font-size:8px;letter-spacing:.22em;
  text-transform:uppercase;color:var(--m);margin:16px 0 7px;
  display:flex;align-items:center;gap:7px}
.sec::after{content:'';flex:1;height:1px;background:var(--b1)}
.dot{width:4px;height:4px;border-radius:50%;background:var(--acc);flex-shrink:0}

/* grids */
.g2{display:grid;grid-template-columns:1fr 1fr;gap:12px}
@media(max-width:900px){.g2{grid-template-columns:1fr}}

/* card */
.card{background:var(--s1);border:1px solid var(--b1);border-radius:7px;overflow:hidden}
.ch{background:var(--s2);padding:6px 11px;border-bottom:1px solid var(--b1);
  font-family:var(--fb);font-weight:700;font-size:9px;text-transform:uppercase;
  letter-spacing:.12em;color:var(--m);display:flex;align-items:center;gap:6px}
.pill{font-size:8px;padding:0 6px;border-radius:20px;font-family:var(--fm);font-weight:400}
.pill-g{background:rgba(34,197,94,.15);color:var(--pos)}
.pill-r2{background:rgba(244,63,94,.15);color:var(--neg)}
.pill-a{background:rgba(245,158,11,.15);color:var(--acc)}
.pill-b2{background:rgba(56,189,248,.15);color:var(--acc2)}
.pill-p{background:rgba(167,139,250,.15);color:var(--pur)}

/* tables */
table{width:100%;border-collapse:collapse}
thead th{padding:5px 7px;text-align:left;font-size:8px;letter-spacing:.08em;
  text-transform:uppercase;color:var(--m);border-bottom:1px solid var(--b1);
  background:var(--s2);white-space:nowrap}
th.nr{text-align:right}
tbody tr{border-bottom:1px solid rgba(26,32,53,.6);transition:background .1s}
tbody tr:last-child{border-bottom:none}
tbody tr:hover{background:rgba(255,255,255,.025)}
td{padding:5px 7px;vertical-align:middle;white-space:nowrap}
td.nr{text-align:right} td.tk{font-family:var(--fb);font-weight:700;font-size:12px;color:var(--w)}
td.sigs{white-space:normal;min-width:200px} td.rank{color:var(--m);font-size:10px;text-align:right;width:20px;padding-right:3px}
.grow td.tk{color:var(--pos)}.lrow td.tk{color:var(--neg)}
.pos{color:var(--pos)}.neg{color:var(--neg)}.nt{color:var(--m)}.na{color:#252d47}
.badge-ob{background:rgba(244,63,94,.15);color:#f87171;padding:1px 5px;border-radius:3px}
.badge-os{background:rgba(34,197,94,.15);color:#4ade80;padding:1px 5px;border-radius:3px}
.badge-nt{color:var(--m)}
.tag-a{background:rgba(34,197,94,.12);color:#4ade80;padding:0 4px;border-radius:3px;font-size:9px;margin-right:2px;display:inline-block;font-family:var(--fb)}
.tag-b{background:rgba(244,63,94,.12);color:#f87171;padding:0 4px;border-radius:3px;font-size:9px;margin-right:2px;display:inline-block;font-family:var(--fb)}
.pill-b,.pill-r{display:inline-block;font-size:8.5px;padding:0 5px;border-radius:3px;margin:1px 1px 1px 0;font-family:var(--fb);white-space:nowrap}
.pill-b{background:rgba(34,197,94,.12);color:#4ade80}
.pill-r{background:rgba(244,63,94,.12);color:#f87171}

/* score bar */
.sb-wrap{display:flex;align-items:center;gap:6px;min-width:90px}
.sb{height:4px;border-radius:2px}.sl{font-size:10px;color:var(--m)}

/* volume bar */
.vbw{display:flex;align-items:center;gap:7px;min-width:120px}
.vb{height:4px;border-radius:2px;background:linear-gradient(90deg,var(--acc2),rgba(56,189,248,.15));max-width:100px;min-width:3px}
.vl{color:var(--m);font-size:10px}
canvas.sp{display:block}

/* ── MOMENTUM TABLE ── */
.mom-period{font-size:8px;color:var(--m);letter-spacing:.08em;display:block;margin-bottom:2px}
.alpha-row td{background:rgba(167,139,250,.04)}

/* ── HEATMAP ── */
.heatmap-grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(180px,1fr));gap:8px}
.hm-tile{border-radius:8px;padding:12px 14px;cursor:default;
  transition:transform .15s,box-shadow .15s;border:1px solid rgba(255,255,255,.06);
  position:relative;overflow:hidden}
.hm-tile:hover{transform:translateY(-2px);box-shadow:0 8px 24px rgba(0,0,0,.4)}
.hm-name{font-family:var(--fb);font-weight:700;font-size:11px;margin-bottom:4px;line-height:1.2}
.hm-pct{font-family:var(--fh);font-size:28px;line-height:1;letter-spacing:.02em}
.hm-sub{font-size:9px;opacity:.7;margin-top:3px;font-family:var(--fm)}
.hm-row{display:flex;justify-content:space-between;font-size:9px;margin-top:5px;padding-top:5px;
  border-top:1px solid rgba(255,255,255,.08)}
.hm-periods{display:flex;gap:5px;flex-wrap:wrap;margin-top:5px}
.hm-period{font-size:8px;padding:1px 5px;border-radius:3px;background:rgba(255,255,255,.08)}
.hm-count{font-size:9px;opacity:.5;margin-top:4px;font-family:var(--fb)}

/* period selector */
.period-tabs{display:flex;gap:6px;margin-bottom:10px}
.ptab{background:var(--s2);border:1px solid var(--b1);border-radius:5px;
  padding:4px 12px;font-size:10px;cursor:pointer;color:var(--m);
  font-family:var(--fb);font-weight:700;transition:all .15s}
.ptab.active,.ptab:hover{background:rgba(56,189,248,.15);color:var(--acc2);border-color:var(--acc2)}

/* ref grid */
.ref-grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(200px,1fr));gap:10px}
.ref-card{background:var(--s1);border:1px solid var(--b1);border-radius:7px;padding:10px 11px;
  position:relative;overflow:hidden}
.ref-card::before{content:'';position:absolute;top:0;left:0;right:0;height:2px;
  background:linear-gradient(90deg,var(--acc),var(--acc2))}
.ref-head{font-family:var(--fb);font-weight:700;font-size:12px;color:var(--w);margin-bottom:4px}
.ref-ticker{font-family:var(--fm);font-size:9px;color:var(--m);display:block;margin-bottom:6px}
.ref-row{display:flex;align-items:center;gap:5px;padding:3px 0;border-bottom:1px solid rgba(26,32,53,.5)}
.ref-row:last-child{border:none}
.ref-badge{font-size:8px;padding:1px 5px;border-radius:3px;min-width:68px;text-align:center;flex-shrink:0;font-family:var(--fm)}
.ref-desc{color:var(--m);font-size:10px;font-family:var(--fb)}

/* footer */
footer{border-top:1px solid var(--b1);padding:12px 0 0;margin-top:28px;color:var(--m);font-size:10px;line-height:1.9;font-family:var(--fb)}
.fts{color:#1e2640;font-size:9px}
@media(max-width:600px){td,th{padding:4px 5px}}
"""

SPARKLINE_JS = """
document.querySelectorAll('canvas.sp').forEach(cv=>{
  const c=JSON.parse(cv.dataset.c||'[]'),col=cv.dataset.col||'#22c55e';
  if(c.length<2)return;
  const ctx=cv.getContext('2d'),W=cv.width,H=cv.height;
  const mn=Math.min(...c),mx=Math.max(...c),rng=mx-mn||1,p=3;
  const xi=i=>p+(i/(c.length-1))*(W-2*p);
  const yi=v=>H-p-((v-mn)/rng)*(H-2*p);
  const g=ctx.createLinearGradient(0,0,0,H);
  g.addColorStop(0,col+'44');g.addColorStop(1,col+'00');
  ctx.beginPath();c.forEach((v,i)=>i===0?ctx.moveTo(xi(i),yi(v)):ctx.lineTo(xi(i),yi(v)));
  ctx.lineTo(xi(c.length-1),H);ctx.lineTo(xi(0),H);ctx.closePath();ctx.fillStyle=g;ctx.fill();
  ctx.beginPath();c.forEach((v,i)=>i===0?ctx.moveTo(xi(i),yi(v)):ctx.lineTo(xi(i),yi(v)));
  ctx.strokeStyle=col;ctx.lineWidth=1.5;ctx.lineJoin='round';ctx.stroke();
  ctx.beginPath();ctx.arc(xi(c.length-1),yi(c[c.length-1]),2.2,0,Math.PI*2);
  ctx.fillStyle=col;ctx.fill();
});
"""

PERIOD_TAB_JS = """
// Period tab switcher for momentum leaderboard
function switchMomTab(period) {
  document.querySelectorAll('.ptab').forEach(t=>t.classList.remove('active'));
  document.querySelectorAll('[data-period]').forEach(row=>{
    row.style.display = (row.dataset.period === period) ? '' : 'none';
  });
  event.target.classList.add('active');
}
// Show 1M by default
document.addEventListener('DOMContentLoaded',()=>{
  const btn = document.querySelector('.ptab[onclick*="1M"]');
  if(btn) { btn.click(); btn.dispatchEvent(new Event('click')); }
  // manual trigger
  document.querySelectorAll('[data-period]').forEach(row=>{
    row.style.display = row.dataset.period === '1M' ? '' : 'none';
  });
  document.querySelectorAll('.ptab').forEach(t=>{
    if(t.textContent==='1M') t.classList.add('active');
  });
});
"""


# ══════════════════════════════════════════════════════════════════
# HTML BUILD
# ══════════════════════════════════════════════════════════════════

REF_DATA = [
    ("SMA/EMA","Simple & Exponential MAs",[("P>EMA20","Short-term uptrend","badge-os"),("P>EMA50","Medium-term uptrend","badge-os"),("P>EMA200","Long-term bull zone","badge-os"),("E20>E50","Golden cross ↑","badge-os")]),
    ("MACD","12/26/9 Histogram",[("Hist>0","Bull momentum","badge-os"),("Hist<0","Bear momentum","badge-ob"),("Cross ↑","Buy signal","badge-os"),("Cross ↓","Sell signal","badge-ob")]),
    ("RSI","Relative Strength (14)",[("<30","Oversold bounce","badge-os"),("30–50","Below neutral","badge-nt"),("50–70","Bullish zone","badge-nt"),(">70","Overbought","badge-ob")]),
    ("BB","Bollinger Bands %B",[("%B>0.9","Near upper (OB)","badge-ob"),("0.4–0.85","Mid-upper (bull)","badge-nt"),("%B<0.1","Near lower (bounce)","badge-os"),("Squeeze","Breakout imminent","badge-nt")]),
    ("Stoch","Stochastic K/D (14,3)",[("K<20,K>D","Oversold cross ↑","badge-os"),("K>80","Overbought","badge-ob"),("K>50,K>D","Bullish","badge-nt"),("K<50,K<D","Bearish","badge-nt")]),
    ("ATR","Avg True Range (14)",[("High ATR","High volatility","badge-ob"),("Low ATR","Low vol — watch","badge-nt"),("ATR↑","Expanding","badge-nt"),("ATR↓","Contracting","badge-os")]),
    ("OBV","On-Balance Volume",[("Rising","Smart money in","badge-os"),("Falling","Distribution","badge-ob"),("OBV>Price","Bull divergence","badge-os"),("OBV<Price","Bear divergence","badge-ob")]),
    ("VWAP","20-Day Rolling VWAP",[("P>VWAP","Inst. buy zone","badge-os"),("P<VWAP","Inst. sell zone","badge-ob"),("Cross ↑","Buy signal","badge-os"),("Cross ↓","Sell signal","badge-ob")]),
    ("Supertrend","Period 10, Mult 3.0",[("Bullish ↑","Above ST line","badge-os"),("Bearish ↓","Below ST line","badge-ob"),("Flip ↑","Fresh buy","badge-os"),("Flip ↓","Fresh sell","badge-ob")]),
    ("Ichimoku","9/26/52 Cloud",[("Above cloud","Strong bullish","badge-os"),("In cloud","Indecision","badge-nt"),("Below cloud","Strong bearish","badge-ob"),("TK cross ↑","Entry signal","badge-os")]),
    ("Fibonacci","50-bar Retracement",[("0%","Recent high (res.)","badge-ob"),("38.2%","First key support","badge-nt"),("61.8%","Golden ratio","badge-os"),("100%","Recent low (sup.)","badge-os")]),
    ("ADX","Avg Directional (14)",[("<20","No trend / ranging","badge-nt"),("20–25","Emerging trend","badge-nt"),(">25,+DI","Strong uptrend","badge-os"),(">25,−DI","Strong downtrend","badge-ob")]),
]


def build_html(df_all, df_idx, sentiment, sector_stats, n=TOP_N):
    now_str  = datetime.now().strftime("%d %b %Y  %I:%M %p IST")
    date_str = datetime.now().strftime("%a, %d %b %Y")
    df = df_all.dropna(subset=["pct_1d"])

    gainers  = df.nlargest(n,"pct_1d")
    losers   = df.nsmallest(n,"pct_1d")
    top_vol  = df.nlargest(n,"volume")
    top_ta   = df.nlargest(n,"ta_score")
    max_vol  = int(top_vol["volume"].max()) if not top_vol.empty else 1

    g_count=(df["pct_1d"]>0).sum(); l_count=(df["pct_1d"]<0).sum(); u_count=(df["pct_1d"]==0).sum()
    avg_rsi=df["rsi"].dropna().mean(); avg_ta=df["ta_score"].mean()

    bench_str=" · ".join(f"{p}: {v:+.1f}%" for p,v in BENCH_RETURNS.items() if np.isfinite(v))

    # ── Stats strip ──────────────────────────────────────────────
    stats_html=f"""<div class="stats-strip">
      <div class="stat-box"><div class="stat-val" style="color:var(--pos)">{g_count}</div><div class="stat-lbl">Advances</div></div>
      <div class="stat-box"><div class="stat-val" style="color:var(--neg)">{l_count}</div><div class="stat-lbl">Declines</div></div>
      <div class="stat-box"><div class="stat-val" style="color:var(--m)">{u_count}</div><div class="stat-lbl">Unchanged</div></div>
      <div class="stat-box"><div class="stat-val">{len(df)}</div><div class="stat-lbl">Universe</div></div>
      <div class="stat-box"><div class="stat-val">{avg_rsi:.0f}</div><div class="stat-lbl">Avg RSI</div></div>
      <div class="stat-box"><div class="stat-val">{avg_ta:.1f}</div><div class="stat-lbl">Avg TA Score</div></div>
    </div>"""

    # ── Index rows ────────────────────────────────────────────────
    idx_rows=""
    for _,r in df_idx.iterrows():
        idx_rows+=f"""<tr>
          <td class="tk">{r['name']}</td><td class="nr">{r['last']:,.2f}</td>
          <td class="nr">{pct_span(r['pct_1d'])}</td><td class="nr">{pct_span(r['pct_5d'])}</td>
          <td class="nr">{pct_span(r.get('mom_1m',np.nan))}</td>
          <td class="nr">{pct_span(r.get('mom_3m',np.nan))}</td>
          <td class="nr">{pct_span(r.get('mom_1y',np.nan))}</td>
          <td class="nr">{rsi_badge(r['rsi'])}</td>
          <td class="nr">{adx_badge(r['adx'],np.nan,np.nan)}</td>
          <td class="nr">{supertrend_badge(r['supertrend_sig'])}</td>
          <td>{sparkline(r['closes'])}</td>
        </tr>"""

    # ── Movers ────────────────────────────────────────────────────
    def mover_rows(dff, cls=""):
        out=""
        for i,(_,r) in enumerate(dff.iterrows(),1):
            sym=r['ticker'].replace('.NS','')
            out+=f'<tr class="{cls}"><td class="rank">{i}</td><td class="tk">{sym}</td><td class="nr">&#8377;{r["last"]:,.2f}</td><td class="nr">{pct_span(r["pct_1d"])}</td><td class="nr">{r["volume"]:,}</td></tr>'
        return out

    # ── Volume rows ───────────────────────────────────────────────
    vol_rows=""
    for i,(_,r) in enumerate(top_vol.iterrows(),1):
        sym=r['ticker'].replace('.NS','')
        vol_rows+=f'<tr><td class="rank">{i}</td><td class="tk">{sym}</td><td class="nr">&#8377;{r["last"]:,.2f}</td><td class="nr">{pct_span(r["pct_1d"])}</td><td>{vol_bar(int(r["volume"]),max_vol)}</td></tr>'

    # ── Multi-period Momentum tables (tabbed) ─────────────────────
    PERIODS = [("1D","1 Day"),("1W","1 Week"),("1M","1 Month"),("3M","3 Months (Quarterly)"),("1Y","1 Year (Annual)")]
    mom_tab_html = '<div class="period-tabs">'
    for pid,plbl in PERIODS:
        active = "active" if pid=="1M" else ""
        mom_tab_html += f'<button class="ptab {active}" onclick="switchMomTab(\'{pid}\')">{pid}</button>'
    mom_tab_html += "</div>"

    # Build a combined table; rows tagged data-period for JS filtering
    mom_rows=""
    for pid,_ in PERIODS:
        col=f"mom_{pid.lower()}"; rel_col=f"rel_{pid.lower()}"
        if pid=="1D": col="mom_1d"; rel_col="rel_1d"
        top_mom=df.dropna(subset=[col]).nlargest(n,col)
        for i,(_,r) in enumerate(top_mom.iterrows(),1):
            sym=r['ticker'].replace('.NS','')
            mom_v=r.get(col,np.nan); rel_v=r.get(rel_col,np.nan)
            bench_v=BENCH_RETURNS.get(pid,np.nan)
            style='display:none' if pid!="1M" else ''
            mom_rows+=f"""<tr data-period="{pid}" style="{style}">
              <td class="rank">{i}</td><td class="tk">{sym}</td>
              <td class="nr">&#8377;{r['last']:,.2f}</td>
              <td class="nr">{pct_span(mom_v)}</td>
              <td class="nr">{rel_span(rel_v)}</td>
              <td class="nr">{pct_span(bench_v)}</td>
              <td class="nr">{rsi_badge(r['rsi'])}</td>
              <td class="nr">{adx_badge(r['adx'],r['adx_di_plus'],r['adx_di_minus'])}</td>
              <td class="nr">{supertrend_badge(r['supertrend_sig'])}</td>
              <td>{sparkline(r['closes_5d'])}</td>
            </tr>"""
        # Also build worst-performers (bottom 5 per period)
        bot_mom=df.dropna(subset=[col]).nsmallest(min(5,n),col)
        for i,(_,r) in enumerate(bot_mom.iterrows(),1):
            sym=r['ticker'].replace('.NS','')
            mom_v=r.get(col,np.nan); rel_v=r.get(rel_col,np.nan)
            bench_v=BENCH_RETURNS.get(pid,np.nan)
            style='display:none' if pid!="1M" else ''
            mom_rows+=f"""<tr data-period="{pid}" class="lrow" style="{style}">
              <td class="rank" style="color:var(--neg)">▼{i}</td><td class="tk">{sym}</td>
              <td class="nr">&#8377;{r['last']:,.2f}</td>
              <td class="nr">{pct_span(mom_v)}</td>
              <td class="nr">{rel_span(rel_v)}</td>
              <td class="nr">{pct_span(bench_v)}</td>
              <td class="nr">{rsi_badge(r['rsi'])}</td>
              <td class="nr">{adx_badge(r['adx'],r['adx_di_plus'],r['adx_di_minus'])}</td>
              <td class="nr">{supertrend_badge(r['supertrend_sig'])}</td>
              <td>{sparkline(r['closes_5d'])}</td>
            </tr>"""

    # ── TA Picks ──────────────────────────────────────────────────
    ta_rows=""
    for i,(_,r) in enumerate(top_ta.iterrows(),1):
        sym=r['ticker'].replace('.NS','')
        ichi=r.get('ichimoku',{})
        ta_rows+=f"""<tr>
          <td class="rank">{i}</td><td class="tk">{sym}</td>
          <td class="nr">&#8377;{r['last']:,.2f}</td>
          <td>{score_bar(r['ta_score'])}</td>
          <td class="nr">{rsi_badge(r['rsi'])}</td>
          <td class="nr">{macd_badge(r['macd_hist'])}</td>
          <td class="nr">{bb_badge(r['bb_pctb'])}</td>
          <td class="nr">{stoch_badge(r['stoch_k'],r['stoch_d'])}</td>
          <td class="nr">{adx_badge(r['adx'],r['adx_di_plus'],r['adx_di_minus'])}</td>
          <td class="nr">{supertrend_badge(r['supertrend_sig'])}</td>
          <td class="nr">{ichimoku_badge(ichi)}</td>
          <td class="nr">{obv_badge(r['obv_trend'])}</td>
          <td class="nr">{vwap_badge(r['last'],r['vwap'])}</td>
          <td class="nr" style="font-size:10px;color:var(--m)">{r.get('fib_nearest','—')}</td>
          <td>{ema_stack(r['last'],r['ema20'],r['ema50'],r['ema200'])}</td>
          <td class="sigs">{signal_pills(r['ta_signals'])}</td>
        </tr>"""

    # ── Sector Heatmap ────────────────────────────────────────────
    sorted_sectors=sorted(sector_stats.items(),key=lambda kv:kv[1]["avg_1d"],reverse=True)
    heatmap_html=""
    for sector, stats in sorted_sectors:
        avg1d=stats["avg_1d"]; avg1w=stats["avg_1w"]
        avg1m=stats["avg_1m"]; avg3m=stats["avg_3m"]; avg1y=stats["avg_1y"]
        style=heatmap_style(avg1d)
        sign=lambda v: f'+{v:.1f}%' if v>0 else f'{v:.1f}%' if np.isfinite(v) else '—'
        heatmap_html+=f"""<div class="hm-tile" style="{style}">
          <div class="hm-name">{sector}</div>
          <div class="hm-pct">{sign(avg1d)}</div>
          <div class="hm-count">{stats['count']} stocks</div>
          <div class="hm-periods">
            <span class="hm-period">1W: {sign(avg1w)}</span>
            <span class="hm-period">1M: {sign(avg1m)}</span>
            <span class="hm-period">3M: {sign(avg3m)}</span>
            <span class="hm-period">1Y: {sign(avg1y)}</span>
          </div>
          <div class="hm-row">
            <span>&#9650; {stats.get('best_1d','—')}</span>
            <span>&#9660; {stats.get('worst_1d','—')}</span>
          </div>
        </div>"""

    # ── TA Reference ──────────────────────────────────────────────
    ref_html=""
    for title,sub,levels in REF_DATA:
        inner="".join(f'<div class="ref-row"><span class="{cls} ref-badge">{val}</span><span class="ref-desc">{desc}</span></div>' for val,desc,cls in levels)
        ref_html+=f'<div class="ref-card"><div class="ref-head">{title}</div><span class="ref-ticker">{sub}</span>{inner}</div>'

    return f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>NSE Nifty 500 Dashboard v5</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;500&family=Bebas+Neue&family=Manrope:wght@400;600;700&display=swap" rel="stylesheet">
<style>{CSS}</style>
</head>
<body><div class="wrap">

<header>
  <div>
    <div class="logo">NSE <em>Nifty 500</em> <b>Dashboard</b> <span style="font-size:13px;color:var(--m);letter-spacing:.1em">v5</span></div>
    <div class="hdate">{date_str} &nbsp;·&nbsp; 13 Indicators &nbsp;·&nbsp; Multi-Period Momentum &nbsp;·&nbsp; Sectoral Heatmap &nbsp;·&nbsp; {len(df_all)} stocks</div>
    <div style="margin-top:5px;font-size:10px;color:var(--m);font-family:var(--fb)">
      Nifty 500 benchmark: <span style="color:var(--acc2)">{bench_str}</span>
    </div>
  </div>
  <div class="sent"><b>&#9654; Market Pulse</b>{sentiment}</div>
</header>

{stats_html}

<!-- ══ SECTORAL HEATMAP ══ -->
<div class="sec"><span class="dot"></span>Sectoral Heatmap — Average Change % (20 Sectors · sorted by 1D performance)</div>
<div class="heatmap-grid">{heatmap_html}</div>

<!-- ══ INDICES ══ -->
<div class="sec"><span class="dot"></span>Key Indices — Multi-Period Returns</div>
<div class="card" style="overflow-x:auto"><table>
  <thead><tr>
    <th>Index</th><th class="nr">Close</th>
    <th class="nr">1D%</th><th class="nr">1W%</th><th class="nr">1M%</th>
    <th class="nr">3M%</th><th class="nr">1Y%</th>
    <th class="nr">RSI</th><th class="nr">ADX</th><th class="nr">Supertrend</th><th>Trend</th>
  </tr></thead>
  <tbody>{idx_rows}</tbody>
</table></div>

<!-- ══ MOVERS ══ -->
<div class="sec"><span class="dot"></span>Top {n} Daily Movers</div>
<div class="g2">
  <div class="card">
    <div class="ch">&#9650; Top {n} Gainers <span class="pill pill-g">LONG</span></div>
    <table><thead><tr><th>#</th><th>Ticker</th><th class="nr">Close</th><th class="nr">1D%</th><th class="nr">Volume</th></tr></thead>
    <tbody>{mover_rows(gainers,'grow')}</tbody></table>
  </div>
  <div class="card">
    <div class="ch">&#9660; Bottom {n} Losers <span class="pill pill-r2">SHORT</span></div>
    <table><thead><tr><th>#</th><th>Ticker</th><th class="nr">Close</th><th class="nr">1D%</th><th class="nr">Volume</th></tr></thead>
    <tbody>{mover_rows(losers,'lrow')}</tbody></table>
  </div>
</div>

<!-- ══ MULTI-PERIOD MOMENTUM ══ -->
<div class="sec"><span class="dot"></span>Multi-Period Momentum — Absolute vs Index (Nifty 500 alpha · Top {n} + Bottom 5)</div>
<div class="card" style="overflow-x:auto">
{mom_tab_html}
<table>
  <thead><tr>
    <th>#</th><th>Ticker</th><th class="nr">Close</th>
    <th class="nr">Return</th><th class="nr">α vs N500</th><th class="nr">N500 Bench</th>
    <th class="nr">RSI</th><th class="nr">ADX</th><th class="nr">Supertrend</th><th>5D Trend</th>
  </tr></thead>
  <tbody>{mom_rows}</tbody>
</table>
</div>

<!-- ══ VOLUME ══ -->
<div class="sec"><span class="dot"></span>Top {n} Volume Leaders</div>
<div class="card"><table>
  <thead><tr><th>#</th><th>Ticker</th><th class="nr">Close</th><th class="nr">1D%</th><th>Relative Volume</th></tr></thead>
  <tbody>{vol_rows}</tbody>
</table></div>

<!-- ══ TECHNICAL PICKS ══ -->
<div class="sec"><span class="dot"></span>Technical Picks — Top {n} by TA Score (13 Indicators · max 18 pts)</div>
<div class="card" style="overflow-x:auto"><table>
  <thead><tr>
    <th>#</th><th>Ticker</th><th class="nr">Close</th><th>Score</th>
    <th class="nr">RSI</th><th class="nr">MACD</th><th class="nr">BB%B</th>
    <th class="nr">Stoch</th><th class="nr">ADX</th><th class="nr">Supertrend</th>
    <th class="nr">Ichimoku</th><th class="nr">OBV</th><th class="nr">VWAP</th>
    <th class="nr">Fib</th><th>EMA Stack</th><th>Signals</th>
  </tr></thead>
  <tbody>{ta_rows}</tbody>
</table></div>

<!-- ══ TA REFERENCE ══ -->
<div class="sec"><span class="dot"></span>Indicator Quick Reference — 13 Indicators</div>
<div class="ref-grid">{ref_html}</div>

<footer>
  &#9888; Data sourced from Yahoo Finance (yfinance). For <strong>informational purposes only</strong>.
  Not investment advice. Consult a SEBI-registered advisor before trading. Past performance is not indicative of future results.<br>
  <span class="fts">Generated: {now_str}</span>
</footer>
</div>
<script>{SPARKLINE_JS}</script>
<script>{PERIOD_TAB_JS}</script>
</body></html>"""


# ══════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════

def main():
    print("\n╔══════════════════════════════════════════════════════════════╗")
    print("║  NSE Nifty 500 Trading Dashboard  v5                        ║")
    print("║  Multi-Period Momentum · Alpha vs Index · Sectoral Heatmap  ║")
    print("╚══════════════════════════════════════════════════════════════╝\n")
    print(f"  Universe : {len(NIFTY_500)} tickers  |  Workers : {MAX_WORKERS}  |  Period : {TECH_PERIOD}\n")

    t0=time.time()

    print("▸ Fetching Nifty 500 benchmark returns…")
    fetch_benchmark_returns()

    print("▸ Fetching Nifty 500 stocks (parallel)…")
    df_stocks = fetch_all_parallel(list(NIFTY_500))
    print(f"  ✓ {len(df_stocks)} stocks in {time.time()-t0:.0f}s")

    print("▸ Fetching 5 indices…")
    df_idx = fetch_indices(NSE_INDICES)

    print("▸ Computing sector stats…")
    sector_stats = build_sector_stats(df_stocks)
    print(f"  ✓ {len(sector_stats)} sectors")

    sentiment = compute_sentiment(df_stocks)

    print("▸ Rendering HTML…")
    html = build_html(df_stocks, df_idx, sentiment, sector_stats, n=TOP_N)

    out="nse_dashboard.html"
    with open(out,"w",encoding="utf-8") as f:
        f.write(html)

    elapsed=time.time()-t0
    top10=df_stocks.nlargest(TOP_N,"ta_score")[["ticker","ta_score","mom_1d","mom_1m","mom_3m","mom_1y","rel_1m","adx","supertrend_sig"]]
    print(f"\n✅  Saved → {out}  ({elapsed:.0f}s total)\n")
    print(f"  Top {TOP_N} TA Picks:\n{top10.to_string(index=False)}\n")


if __name__ == "__main__":
    main()

In [0]:
!ls
!cp nse_dashboard.html /Workspace/Users/theoldstoryteller001@gmail.com/new/index.html

In [0]:
import shutil
import subprocess

repo_dir='/Workspace/Repos/git/new/'
# Copy nse_dashboard.html to the repo
shutil.copy("AI_dashboard.ipynb", "/Workspace/Repos/git/new/")
subprocess.run(["git", "add", "sd nse_dashboard.html"], cwd='/Workspace/Repos/git/new/')
subprocess.run(["git", "commit", "-m", "Update index.html with latest dashboard"], cwd=repo_dir)
subprocess.run(["git", "push"], cwd=repo_dir)


In [0]:
subprocess.run(["git", "init"], cwd='/Workspace/Repos/git/new/')

In [0]:
import shutil
import subprocess

# Change directory to the repo
repo_dir = "/Workspace/Repos/git/new/"
subprocess.run(["git", "add", "sd nse_dashboard.html"], cwd='/Workspace/Repos/git/new/')
# subprocess.run(["git", "commit", "-m", "Update index.html with latest dashboard"], cwd=repo_dir)
# subprocess.run(["git", "push"], cwd=repo_dir)